# Making Sense of a Black Box

**Notebook 4 of 4.** A boosted model of 400 trees predicts well and cannot be read the way
a single tree or a set of coefficients can. This notebook asks three different questions of
exactly that model, and the three answers disagree — which is the point.

Nothing is trained here: the saved model is loaded below.

## 1. The model

`models/xgboost_tuned.json` holds the XGBoost regressor the Optuna search settled on: 400
trees, `max_depth=8`, fitted on 16,512 California districts to predict `MedHouseVal`, the
median house value in units of **$100,000**.

Saved in XGBoost's own JSON format rather than as a pickle: it survives a library upgrade,
and loading it cannot execute code the way unpickling an untrusted file can.

In [ ]:
import warnings

# shap imports tqdm, which warns about ipywidgets on a plain kernel. It fires on import,
# so the filter goes first
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance, partial_dependence
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor

import shap

RSEED = 42
N_JOBS = 4

In [ ]:
df = pd.read_csv("data/california_housing.csv")

y = df["MedHouseVal"]
X = pd.get_dummies(df.drop(columns="MedHouseVal"), columns=["location"]).astype(float)

# the same split the model was fitted with, so the test districts really are unseen
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RSEED
)

model = XGBRegressor()
model.load_model("models/xgboost_tuned.json")

# n_estimators is not restored into a bare XGBRegressor; ask the booster itself
print(f"loaded a model of {model.get_booster().num_boosted_rounds()} trees "
      f"over {X.shape[1]} columns")
print(f"test RMSE {root_mean_squared_error(y_test, model.predict(X_test)):.4f}")

## 2. The importance the model hands you for free

Every tree model exposes `feature_importances_`. For XGBoost it is the **gain**: for each
column, the average drop in training loss across the splits that used it — the average, not
the total over the 400 trees — then rescaled so the twelve numbers **sum to 1**. Each one is
a *share* of the total gain, not an amount of loss, so read them against each other and
never as RMSE.

It costs nothing, which is the problem — it is the number people quote without asking what
it measures.

In [ ]:
gain = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(gain.round(4).to_string())

**`location_inland` comes first, taking 0.389 of the total gain — nearly three times
`MedInc`'s share.** Three of the top four are one-hot location columns, and the two
coordinates sit fifth and sixth.

Notice what gain counts: the loss a column removed **at the moments it was split on**, on
the training data. A binary column that separates the state near the root of every tree
collects a lot of credit in one go. Whether the **predictions** depend on it is a different
question, and gain never asks it.

## 3. Permutation importance: break a column and see what happens

The question gain does not ask is the direct one. **Shuffle one column** so it keeps its
distribution but loses its relationship to the target, predict again, and see how much worse
the error gets. A column the model leans on will hurt when scrambled; one it ignores will
not.

It is measured here on the **test** districts, because the question is what the model relies
on when predicting data it has not seen.

In [ ]:
# n_repeats=5: each column is shuffled five times, since one shuffle is one draw of luck
result = permutation_importance(
    model, X_test, y_test, n_repeats=5, random_state=RSEED,
    scoring="neg_root_mean_squared_error", n_jobs=N_JOBS,
)

perm = pd.DataFrame(
    {"rise in RMSE": result.importances_mean, "sd": result.importances_std},
    index=X.columns,
).sort_values("rise in RMSE", ascending=False)
print(perm.round(4).to_string())

**`Longitude` first at 0.487, `Latitude` second at 0.464, `MedInc` third at 0.318** — and
`location_inland`, which gain put first, falls to **sixth at 0.055**.

Scrambling the longitude costs nearly $49,000 of error per district; scrambling
`location_inland` costs a ninth as much. The predictions are built on **where the district
is and what it earns**, and the location dummies are a coarse summary of coordinates the
model already has. Every standard deviation is under 0.007, so the ordering is not noise.

## 4. Partial dependence: which direction, and how much

Both measures so far rank columns. Neither says **what the model does with one** — whether
house values rise or fall with income, and by how much.

**Partial dependence** answers that. For a grid of values of one column, set *every*
district to that value, predict, and average. What is left is the model's response to that
column with the others averaged out.

In [ ]:
def dependence(column, resolution=20):
    """Partial dependence of the prediction on one column.

    Args:
        column: the column name to vary.
        resolution: how many points across its range.

    Returns:
        tuple: (grid values, average prediction at each).
    """
    result = partial_dependence(
        model, X_train, [list(X.columns).index(column)], grid_resolution=resolution
    )
    return result["grid_values"][0], result["average"][0]


fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, column in zip(axes, ["MedInc", "AveOccup", "HouseAge"]):
    grid, average = dependence(column)
    ax.plot(grid, average, color="#c8102e")
    ax.set_title(f"{column}   swing {average.max() - average.min():.3f}")
    ax.set_xlabel(column)
    print(f"{column:<10} varies from {grid.min():.2f} to {grid.max():.2f}, "
          f"prediction {average.min():.3f} to {average.max():.3f}, "
          f"swing {average.max() - average.min():.3f}")
# one shared y range, so the three curves are compared rather than each rescaled to fill
axes[0].set_ylabel("predicted value ($100,000s)")
for ax in axes:
    ax.set_ylim(1.5, 3.2)
fig.tight_layout()
plt.show()

Read the **swing**, the distance from the lowest point of each curve to the highest:

- **`MedInc` 1.426** — $142,600 across the income range, climbing steadily. This is the
  relationship doing the work;
- **`AveOccup` 0.762**, falling: more people per household predicts a lower value;
- **`HouseAge` 0.206**, nearly flat, agreeing with permutation importance putting it seventh.

One caution applies to every such plot: setting every district's income to $80,000 while
leaving its location untouched produces districts that **do not exist**, and the curve
averages the model's answers about them. It describes the model, not the world.

## 5. SHAP: what moved *this* prediction

Everything so far describes the model over the whole dataset. Neither answers the question
you get asked about an individual case: *why did it predict that for this district?*

**SHAP** assigns each column a number for each row — its contribution to that row's
prediction, in the target's own units. The contributions have a property the other two
measures do not:

$$\text{prediction} = \text{base value} + \sum_{j} \text{SHAP}_j$$

where the **base value** is the mean prediction over the data, and $\text{SHAP}_j$ is column
$j$'s contribution for this row. The attribution is **complete**: the parts add up to the
whole, exactly.

For tree models this is computed exactly and quickly by `shap.TreeExplainer`, rather than
approximated.

> **Exercise.** Build a `shap.TreeExplainer` on the model, compute SHAP values for the first
> 1,000 test districts, and print the mean absolute SHAP value per column, sorted. Then check
> the equation above holds: compare `explainer.expected_value + shap_values.sum(axis=1)`
> against the model's own predictions and print the largest difference.

<details><summary>
Click here for a hint...
</summary>

`explainer.shap_values(sample)` returns an array of one row per district and one column per
feature. `np.abs(...).mean(axis=0)` collapses it to one number per column.

</details>

In [ ]:
# your code: a TreeExplainer, SHAP values for the first 1,000 test districts, the mean
# absolute value per column, and a check that base + contributions = prediction

<details><summary>
Click here to compare your answer...
</summary>

| column | mean \|SHAP\| |
| :-- | --: |
| `MedInc` | 0.3334 |
| `Latitude` | 0.2672 |
| `Longitude` | 0.2497 |
| `AveOccup` | 0.1786 |
| `AveRooms` | 0.1466 |
| `location_inland` | 0.1457 |

The base value is **2.0717**, the mean prediction, and the largest reconstruction gap is
**about 1e-05** — floating point, not approximation.

That exactness is what the other two cannot offer. A permutation importance of 0.487 is a
statement about a column across the dataset; it cannot be handed to anyone asking about one
district.

</details>

### One district, explained

The same values read along a single row instead of down a column.

In [ ]:
# rebuilt here rather than taken from the exercise above, so this cell and section 6
# run whether or not the exercise has been done
explainer = shap.TreeExplainer(model)
sample = X_test.iloc[:1000]
shap_values = explainer.shap_values(sample)
mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns)

district = 0
contributions = pd.Series(shap_values[district], index=X.columns)
# the six that moved this prediction furthest, in either direction
top = contributions.reindex(contributions.abs().sort_values(ascending=False).index)[:6][::-1]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(top.index, top.values,
        color=["#c8102e" if v < 0 else "#2a6ebb" for v in top.values])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("contribution to the prediction ($100,000s)")
fig.tight_layout()
plt.show()

print(f"base value        {explainer.expected_value:.3f}")
print(f"+ contributions   {shap_values[district].sum():+.3f}")
print(f"= prediction      {model.predict(sample)[district]:.3f}")
print(f"actual value      {y_test.iloc[district]:.3f}")
print("\nthe six that moved it furthest:")
print(top[::-1].round(3).to_string())

The model starts from the base value **2.072**, the average district, and this one ends at
**0.529**. Every one of the six pushed it down: `MedInc` by **−0.436**, `Latitude` by −0.266,
`location_inland` by −0.247, `AveOccup` by −0.194.

So the answer to *why so low* is a sentence: a low-income inland district, and the income is
the largest single reason. The actual value is 0.477, so the model was close.

## 6. The three answers, side by side

Each measure was given the same model and the same data, and each asked it a different
question.

In [ ]:
ranks = pd.DataFrame({
    "gain": gain,
    "permutation": perm["rise in RMSE"],
    "mean |SHAP|": mean_abs,
}).rank(ascending=False).astype(int)

print(ranks.sort_values("permutation").to_string())

**`location_inland` is 1st by gain and 6th by both others. `Longitude` is 5th by gain and
1st by permutation.** The disagreement is not a defect in any of them; they measure
different things:

- **gain** — how much this column reduced the training loss when it was split on;
- **permutation importance** — how much worse the test predictions get without it;
- **mean \|SHAP\|** — how far this column moves a prediction, on average, per district.

Which to reach for follows from the question:

- *does this column matter for performance?* — permutation importance, measured on held-out
  data;
- *which way does it push, and how much?* — partial dependence;
- *why this prediction?* — SHAP, the only one that answers about a single row.

Gain is the one to be most careful with. It is free, it is what gets quoted, and here it
points at a column the model barely relies on.

## Summary

- XGBoost's free **gain** importance put `location_inland` first; **permutation importance**
  on held-out districts put it sixth, behind both coordinates and income. Gain measures
  training splits, not predictive reliance.
- **Partial dependence** gives direction and size: `MedInc` swings the prediction by
  **1.426**, about $142,600, against `HouseAge`'s 0.206. It averages over rows that may not
  exist, so it describes the model rather than the world.
- **SHAP** attributes one prediction to its columns exactly, base plus contributions
  reproducing it to **1e-05**, and is the only one of the three that explains a **single**
  district.
- The three rankings disagree, and each is right about its own question. Decide the question
  first.

## References & Further Reading

- [**scikit-learn: Permutation importance**](https://scikit-learn.org/stable/modules/permutation_importance.html): Why impurity- and gain-based importances mislead, especially for high-cardinality and correlated columns.
- [**scikit-learn: Partial dependence and ICE**](https://scikit-learn.org/stable/modules/partial_dependence.html): The averaging the curve performs, and individual conditional expectation plots, which show one line per row instead.
- [**Lundberg and Lee, A Unified Approach to Interpreting Model Predictions (2017)**](https://papers.nips.cc/paper/2017/hash/8a20a8621978632d76c43dfd28b67767-Abstract.html): SHAP, and the completeness property section 5 checks.
- [**Lundberg et al., From Local Explanations to Global Understanding with Explainable AI for Trees (2020)**](https://doi.org/10.1038/s42256-019-0138-9): The exact algorithm `TreeExplainer` uses, which is what makes SHAP affordable on 400 trees.
- [**SHAP documentation**](https://shap.readthedocs.io/en/latest/): The library's own plots, including the beeswarm and waterfall views of the values computed here.
- [**Molnar, Interpretable Machine Learning**](https://christophm.github.io/interpretable-ml-book/): Book-length treatment of all three methods, with their failure modes under correlated features.